# ALQAC 2026 - Independent Auditor Prediction & Ensemble Pipeline

Notebook này được thiết kế chuyên biệt để giải quyết sự cố của Auditor Model:
- **Nguyên nhân sự cố trước đây:** Model Primary (Qwen3.5-9B) chạy thành công và ra phân phối đa dạng nhãn, nhưng Auditor bị dính ngoại lệ `AttributeError` ở tầng `generate_json` (do `apply_chat_template` trả về structure không khớp, thiếu `attention_mask`, hoặc lỗi truy cập `.device` khi load multi-GPU / device_map auto). Ngoại lệ bị khối try-catch bắt và fallback gán mặc định nhãn `PARTIAL_A_WIN` cho toàn bộ 50 cases.
- **Giải pháp trong Notebook này:**
  1. **Tách biệt hoàn toàn Auditor:** KHÔNG chạy lại Primary (tiết kiệm thời gian, VRAM và quota Colab). Tự động nạp kết quả Primary đã chạy sẵn từ file `primary_predictions.json` / `submission_primary.json` trong folder `trace/` hoặc `/content/submissions/`.
  2. **Vá lỗi Runtime Auditor triệt để:** Chuẩn hóa `apply_chat_template`, hỗ trợ cả tokenizer và processor, lấy `target_device` an toàn (`next(model.parameters()).device`), thêm `attention_mask`, tự động bóc tách và loại bỏ thinking tokens (`<think>...</think>`), và in traceback chi tiết nếu có lỗi.
  3. **Chạy Auditor độc lập:** Dự đoán nhãn, phân tích claims và trích xuất xác suất thực sự cho Auditor (Aya Expanse 8B hoặc Qwen2.5/Qwen3).
  4. **Ensemble Consensus:** Kết hợp Primary + Auditor bằng thuật toán đồng thuận theo trọng số (Direct + Claims-rule) để tạo ra `submission_ensemble_primary_auditor.json`.
  5. **Đánh giá Ground Truth:** Đo lường Accuracy của Primary, Auditor và Ensemble trên tập Public Test (Ground Truth).

In [ ]:
# =============================================================================
# 1. KIỂM TRA GPU VÀ CÀI ĐẶT THƯ VIỆN CẦN THIẾT
# =============================================================================
!nvidia-smi

# Cài đặt các thư viện cần thiết nếu chạy trên môi trường Colab mới
!pip install -q transformers>=4.48.0 accelerate>=0.34.0 bitsandbytes>=0.43.0


Mon Oct 12 10:35:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             50W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
%%writefile /content/alqac_auditor_standalone.py

from __future__ import annotations

import gc
import json
import math
import os
import re
import shutil
import sys
import time
import traceback
import unicodedata
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Iterable, Optional, Sequence

import numpy as np
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoModelForMultimodalLM,
    AutoProcessor,
    AutoTokenizer,
    BitsAndBytesConfig,
)

# =============================================================================
# Constants & Data Structures
# =============================================================================

LABELS = (
    "A_WIN",
    "PARTIAL_A_WIN",
    "PARTIAL_B_WIN",
    "B_WIN",
)

LABEL_INDEX = {label: index for index, label in enumerate(LABELS)}

LABEL_DEFINITIONS = {
    "A_WIN": "Nguyên đơn A được chấp nhận về cơ bản toàn bộ hoặc phần cốt lõi của yêu cầu chính; B không thắng phản tố hoặc defense tương đương.",
    "PARTIAL_A_WIN": "A và B đều đạt một phần, nhưng A nhận phần cứu trợ chính hoặc có trọng lượng lớn hơn; thường trách nhiệm của B được xác lập nhưng mức/phạm vi yêu cầu của A bị giảm.",
    "PARTIAL_B_WIN": "A chỉ nhận cứu trợ hạn chế hoặc thứ yếu, còn B bác được phần lớn yêu cầu chính hoặc thắng vấn đề/phản tố có trọng lượng lớn hơn.",
    "B_WIN": "Yêu cầu chính của A bị bác về cơ bản/toàn bộ, hoặc vị trí pháp lý chính của B được chấp nhận rõ rệt.",
}

CLAIM_TYPES = {"A_CLAIM", "B_COUNTERCLAIM", "B_DEFENSE", "PROCEDURAL"}
IMPORTANCE_VALUES = {"CORE", "MAJOR", "SECONDARY"}
RESULT_VALUES = {"GRANTED", "PARTIALLY_GRANTED", "REJECTED", "UNCERTAIN"}

@dataclass
class AuditorConfig:
    auditor_model_id: str = "CohereLabs/aya-expanse-8b"
    auditor_fallback_model_id: str = "Qwen/Qwen2.5-7B-Instruct"
    auditor_quant_mode: str = "bf16"
    allow_auditor_fallback: bool = True
    hf_token: Optional[str] = None

    deterministic_generation: bool = True
    auditor_max_new_tokens: int = 950

    max_chunks: int = 24
    max_words_per_chunk: int = 140
    max_total_evidence_words: int = 1900
    max_law_articles: int = 16
    max_words_per_law: int = 150
    max_total_law_words: int = 900
    max_hint_words: int = 230
    max_outcome_clues: int = 10

    direct_primary_weight: float = 0.32
    claim_primary_weight: float = 0.22
    direct_auditor_weight: float = 0.27
    claim_auditor_weight: float = 0.19
    min_valid_claims_for_full_weight: int = 2
    evidence_citation_bonus: float = 0.08
    law_citation_bonus: float = 0.05
    disagreement_claim_rule_bonus: float = 0.08
    expected_private_cases: int = 50

@dataclass
class LawArticle:
    aid: int
    law_id: str
    article_id: str
    title: str
    text: str

@dataclass
class CaseBundle:
    case_id: str
    case_query: str
    packet_text: str
    selected_chunk_ids: list[str]
    selected_law_aids: list[int]
    valid_chunk_ids: set[str]
    valid_law_aids: set[int]

def read_json(path: Path | str) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def atomic_write_json(path: Path | str, payload: Any) -> None:
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    temp = p.with_suffix(p.suffix + ".tmp")
    with open(temp, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    temp.replace(p)

def clean_spaces(text: Any) -> str:
    if not text:
        return ""
    return re.sub(r"\s+", " ", str(text)).strip()

def truncate_words(text: str, max_words: int) -> str:
    tokens = clean_spaces(text).split()
    if len(tokens) <= max_words:
        return " ".join(tokens)
    return " ".join(tokens[:max_words]) + " ..."

def dedupe_preserve(items: Iterable[Any]) -> list[Any]:
    out = []
    seen = set()
    for it in items:
        if it not in seen:
            seen.add(it)
            out.append(it)
    return out

def top_label(probabilities: dict[str, float]) -> str:
    return max(LABELS, key=lambda lbl: probabilities.get(lbl, 0.0))

def normalize_probabilities(
    raw_probs: Optional[dict[str, Any]],
    fallback_label: Optional[str] = None,
) -> dict[str, float]:
    probs = {}
    if isinstance(raw_probs, dict):
        for lbl in LABELS:
            try:
                val = float(raw_probs.get(lbl, 0.0))
                probs[lbl] = max(0.0, val)
            except (TypeError, ValueError):
                probs[lbl] = 0.0
    total = sum(probs.values())
    if total > 1e-6:
        return {lbl: val / total for lbl, val in probs.items()}
    if fallback_label in LABELS:
        return {lbl: (0.70 if lbl == fallback_label else 0.10) for lbl in LABELS}
    return {lbl: 0.25 for lbl in LABELS}

def probability_margin(probabilities: dict[str, float]) -> float:
    ranked = sorted(probabilities.values(), reverse=True)
    if len(ranked) >= 2:
        return float(ranked[0] - ranked[1])
    return 1.0

# =============================================================================
# Corpus and Evidence Loading
# =============================================================================

def load_article_corpus(path: Path) -> dict[int, LawArticle]:
    data = read_json(path)
    corpus = {}
    for law in data:
        law_id = str(law.get("law_id", "")).strip()
        for art in law.get("articles", []):
            try:
                aid = int(art.get("aid"))
            except (TypeError, ValueError):
                continue
            corpus[aid] = LawArticle(
                aid=aid,
                law_id=law_id,
                article_id=str(art.get("article_id", "")).strip(),
                title=clean_spaces(art.get("title", "")),
                text=clean_spaces(art.get("text", "")),
            )
    return corpus

def load_case_retrieval(directory: Path) -> dict[str, dict[str, Any]]:
    out = {}
    for p in sorted(directory.glob("*_evidence_result.json")):
        data = read_json(p)
        case_id = str(data.get("case_id", "")).strip()
        if not case_id:
            case_id = p.stem.replace("_evidence_result", "")
        out[case_id] = data
    return out

def load_private_case_and_law(path: Path) -> dict[str, dict[str, Any]]:
    data = read_json(path)
    return {str(item["case_id"]).strip(): item for item in data}

def build_case_bundles(
    test_cases: Sequence[dict[str, Any]],
    retrieval_by_id: dict[str, dict[str, Any]],
    case_and_law_by_id: dict[str, dict[str, Any]],
    article_by_aid: dict[int, LawArticle],
    config: AuditorConfig,
) -> list[CaseBundle]:
    bundles = []
    for tc in test_cases:
        cid = str(tc["case_id"]).strip()
        c_query = clean_spaces(tc.get("case_query", ""))
        retrieval = retrieval_by_id.get(cid, {})
        cl = case_and_law_by_id.get(cid, {})

        chunks = retrieval.get("retrieved_chunks", [])
        selected_chunks = []
        valid_chunks = set()
        for c in chunks[:config.max_chunks]:
            chid = str(c.get("chunk_id", "")).strip()
            text = clean_spaces(c.get("chunk_text", ""))
            if chid and text:
                selected_chunks.append((chid, truncate_words(text, config.max_words_per_chunk)))
                valid_chunks.add(chid)

        law_aids = dedupe_preserve(cl.get("retrieved_law_aids", []) + retrieval.get("retrieved_law_aids", []))
        selected_laws = []
        valid_laws = set()
        for aid in law_aids[:config.max_law_articles]:
            art = article_by_aid.get(aid)
            if art:
                selected_laws.append(art)
                valid_laws.add(aid)

        evidence_lines = ["[CASE EVIDENCE]"]
        for chid, ctext in selected_chunks:
            evidence_lines.append(f"- {chid}: {ctext}")

        law_lines = ["[LAW EVIDENCE]"]
        for art in selected_laws:
            law_lines.append(f"- AID {art.aid} ({art.law_id} Điều {art.article_id}): {truncate_words(art.text, config.max_words_per_law)}")

        packet = f"CASE ID: {cid}\nCASE QUERY:\n{c_query}\n\n" + "\n".join(evidence_lines) + "\n\n" + "\n".join(law_lines)

        bundles.append(CaseBundle(
            case_id=cid,
            case_query=c_query,
            packet_text=packet,
            selected_chunk_ids=[c[0] for c in selected_chunks],
            selected_law_aids=[art.aid for art in selected_laws],
            valid_chunk_ids=valid_chunks,
            valid_law_aids=valid_laws,
        ))
    return bundles

# =============================================================================
# Robust JSON Extraction Runtime
# =============================================================================

class JsonRuntimeBase:
    @staticmethod
    def extract_json_object(text: str) -> dict[str, Any]:
        cleaned = str(text or "").strip()
        cleaned = re.sub(r"<think>.*?</think>", "", cleaned, flags=re.DOTALL).strip()
        cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned, flags=re.IGNORECASE)
        cleaned = re.sub(r"\s*```$", "", cleaned)

        decoder = json.JSONDecoder()
        for index, char in enumerate(cleaned):
            if char != "{":
                continue
            try:
                value, _ = decoder.raw_decode(cleaned[index:])
                if isinstance(value, dict):
                    return value
            except json.JSONDecodeError:
                continue

        match = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)
        if match:
            try:
                val = json.loads(match.group(0))
                if isinstance(val, dict):
                    return val
            except Exception:
                pass

        raise ValueError(f"Không parse được JSON:\n{cleaned[:1000]}")

# =============================================================================
# Auditor Runtime (Robust & Bug-free)
# =============================================================================

class TextAuditorRuntime(JsonRuntimeBase):
    def __init__(self, config: AuditorConfig):
        self.config = config
        self.model_id = config.auditor_model_id
        self.tokenizer: Optional[Any] = None
        self.processor: Optional[Any] = None
        self.model: Optional[Any] = None
        self.using_fallback = False
        self.device = None

    def _load_model(self, model_id: str) -> None:
        token = self.config.hf_token or os.getenv("HF_TOKEN")
        print(f"Đang tải tokenizer/processor cho {model_id}...")
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(
                model_id,
                token=token,
                trust_remote_code=True,
            )
        except Exception as e:
            print(f"AutoTokenizer warning: {e}, thử AutoProcessor...")
            self.processor = AutoProcessor.from_pretrained(
                model_id,
                token=token,
                trust_remote_code=True,
            )
            self.tokenizer = getattr(self.processor, "tokenizer", self.processor)

        kwargs: dict[str, Any] = {
            "device_map": "auto",
            "torch_dtype": torch.bfloat16,
            "low_cpu_mem_usage": True,
            "trust_remote_code": True,
            "token": token,
        }
        mode = self.config.auditor_quant_mode.lower()
        if mode == "bnb4":
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.bfloat16,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=False,
            )
        elif mode == "bnb8":
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_8bit=True,
            )

        print(f"Đang tải weights model {model_id}...")
        try:
            self.model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
        except Exception as exc:
            print(f"AutoModelForCausalLM không load được ({exc}), thử AutoModelForMultimodalLM...")
            self.model = AutoModelForMultimodalLM.from_pretrained(model_id, **kwargs)

        self.model.eval()
        self.model_id = model_id
        try:
            self.device = next(self.model.parameters()).device
        except Exception:
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def load(self) -> None:
        try:
            self._load_model(self.config.auditor_model_id)
        except Exception as exc:
            if not self.config.allow_auditor_fallback:
                raise
            print(f"Không load được primary auditor ({self.config.auditor_model_id}): {type(exc).__name__}: {exc}")
            print(f"Fallback sang auditor: {self.config.auditor_fallback_model_id}")
            self.tokenizer = None
            self.processor = None
            self.model = None
            gc.collect()
            torch.cuda.empty_cache()
            self.using_fallback = True
            self._load_model(self.config.auditor_fallback_model_id)

        print(f"Auditor loaded successfully: {self.model_id} (fallback={self.using_fallback}, device={self.device})")

    def unload(self) -> None:
        self.tokenizer = None
        self.processor = None
        self.model = None
        gc.collect()
        torch.cuda.empty_cache()

    def generate_json(self, system: str, user: str, max_new_tokens: int) -> dict[str, Any]:
        if self.model is None or (self.tokenizer is None and self.processor is None):
            raise RuntimeError("Auditor runtime chưa load.")

        tok = self.tokenizer or self.processor
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ]

        inputs = None
        for try_fn in [
            lambda: tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt"),
            lambda: tok.apply_chat_template(messages, enable_thinking=False, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt"),
            lambda: tok.apply_chat_template(messages, chat_template_kwargs={"enable_thinking": False}, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt"),
            lambda: tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_tensors="pt"),
        ]:
            try:
                inputs = try_fn()
                if inputs is not None:
                    break
            except Exception:
                continue

        target_device = self.device or (next(self.model.parameters()).device if self.model else "cuda")
        if inputs is None:
            prompt_str = tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
            inputs = tok(prompt_str, return_tensors="pt")

        if isinstance(inputs, torch.Tensor):
            input_ids = inputs.to(target_device)
            attention_mask = torch.ones_like(input_ids).to(target_device)
            gen_inputs = {"input_ids": input_ids, "attention_mask": attention_mask}
            input_length = input_ids.shape[-1]
        elif hasattr(inputs, "items"):
            gen_inputs = {k: v.to(target_device) for k, v in inputs.items() if hasattr(v, "to")}
            input_length = gen_inputs["input_ids"].shape[-1]
        else:
            raise TypeError(f"Chat template output type không hợp lệ: {type(inputs)}")

        pad_id = getattr(tok, "pad_token_id", None) or getattr(tok, "eos_token_id", None) or 0
        eos_id = getattr(tok, "eos_token_id", None) or pad_id

        generation_kwargs = {
            "max_new_tokens": max_new_tokens,
            "do_sample": not self.config.deterministic_generation,
            "pad_token_id": pad_id,
            "eos_token_id": eos_id,
            "use_cache": True,
        }
        if not self.config.deterministic_generation:
            generation_kwargs.update({"temperature": 0.55, "top_p": 0.85})

        with torch.inference_mode():
            generated = self.model.generate(**gen_inputs, **generation_kwargs)

        new_tokens = generated[0, input_length:]
        text = tok.decode(new_tokens, skip_special_tokens=True).strip()
        text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
        return self.extract_json_object(text)

# =============================================================================
# Prompts, Claims Parsing & Consensus Engine
# =============================================================================

AUDITOR_SYSTEM = """/no_think
BẮT ĐẦU NGAY BẰNG DẤU { VÀ CHỈ TRẢ VỀ JSON HỢP LỆ.
Bạn là người phản biện độc lập, không được xác nhận máy móc một mô hình khác.

Hãy tự đánh giá vụ án dân sự Việt Nam từ raw case evidence và law evidence:
- A là nguyên đơn, B là bị đơn;
- xác định yêu cầu cốt lõi, phản tố và defense;
- phân biệt thắng toàn bộ với thắng một phần;
- PARTIAL_A_WIN khi A nhận cứu trợ có trọng lượng lớn hơn;
- PARTIAL_B_WIN khi B thắng vấn đề trung tâm nhưng A vẫn nhận cứu trợ hạn chế;
- B_WIN khi yêu cầu chính của A bị bác về cơ bản;
- không coi hints do model trước sinh là sự thật;
- không coi ý kiến Viện kiểm sát/cấp dưới là kết quả cuối;
- chỉ trả JSON."""

def label_definition_text() -> str:
    return "\n".join(f"- {lbl}: {LABEL_DEFINITIONS[lbl]}" for lbl in LABELS)

def prediction_schema() -> str:
    return """
Schema:
{
  "prediction": "A_WIN|PARTIAL_A_WIN|PARTIAL_B_WIN|B_WIN",
  "probabilities": {
    "A_WIN": 0.0,
    "PARTIAL_A_WIN": 0.0,
    "PARTIAL_B_WIN": 0.0,
    "B_WIN": 0.0
  },
  "claims": [
    {
      "claim_id": "c1",
      "claim_type": "A_CLAIM|B_COUNTERCLAIM|B_DEFENSE|PROCEDURAL",
      "claim": "mô tả ngắn",
      "importance": "CORE|MAJOR|SECONDARY",
      "result": "GRANTED|PARTIALLY_GRANTED|REJECTED|UNCERTAIN",
      "case_evidence_ids": ["chunk_id"],
      "law_aids": [123],
      "short_reason": "tối đa 55 từ"
    }
  ],
  "confidence": "HIGH|MEDIUM|LOW",
  "decision_summary": "tối đa 120 từ"
}
Các xác suất phải không âm và có tổng xấp xỉ 1.
"""

def validate_claim(claim: dict[str, Any], case: CaseBundle) -> dict[str, Any]:
    claim_type = str(claim.get("claim_type", "")).upper()
    if claim_type not in CLAIM_TYPES:
        claim_type = "PROCEDURAL"
    importance = str(claim.get("importance", "")).upper()
    if importance not in IMPORTANCE_VALUES:
        importance = "SECONDARY"
    result = str(claim.get("result", "")).upper()
    if result not in RESULT_VALUES:
        result = "UNCERTAIN"

    evidence_ids = [str(v) for v in claim.get("case_evidence_ids", []) if str(v) in case.valid_chunk_ids]
    law_aids = []
    for v in claim.get("law_aids", []):
        try:
            aid = int(v)
            if aid in case.valid_law_aids:
                law_aids.append(aid)
        except (TypeError, ValueError):
            continue

    return {
        "claim_id": str(claim.get("claim_id", "")),
        "claim_type": claim_type,
        "claim": truncate_words(claim.get("claim", ""), 45),
        "importance": importance,
        "result": result,
        "case_evidence_ids": dedupe_preserve(evidence_ids),
        "law_aids": dedupe_preserve(law_aids),
        "short_reason": truncate_words(claim.get("short_reason", ""), 60),
    }

def parse_model_result(raw: dict[str, Any], case: CaseBundle, model_id: str) -> dict[str, Any]:
    prediction = str(raw.get("prediction", "")).upper()
    if prediction not in LABELS:
        prediction = None
    probabilities = normalize_probabilities(raw.get("probabilities"), prediction)
    if prediction is None:
        prediction = top_label(probabilities)

    claims_raw = raw.get("claims", [])
    if not isinstance(claims_raw, list):
        claims_raw = []
    claims = [validate_claim(it, case) for it in claims_raw if isinstance(it, dict)][:12]

    confidence = str(raw.get("confidence", "LOW")).upper()
    if confidence not in {"HIGH", "MEDIUM", "LOW"}:
        confidence = "LOW"

    return {
        "case_id": case.case_id,
        "model_id": model_id,
        "prediction": prediction,
        "probabilities": probabilities,
        "claims": claims,
        "confidence": confidence,
        "decision_summary": truncate_words(raw.get("decision_summary", ""), 130),
    }

def claim_table_probabilities(claims: list[dict[str, Any]]) -> dict[str, float]:
    if not claims:
        return {lbl: 0.25 for lbl in LABELS}
    weights = {"CORE": 3.0, "MAJOR": 2.0, "SECONDARY": 1.0}
    scores = {"A_WIN": 0.1, "PARTIAL_A_WIN": 0.2, "PARTIAL_B_WIN": 0.2, "B_WIN": 0.1}
    for c in claims:
        w = weights.get(c.get("importance", "SECONDARY"), 1.0)
        ctype = c.get("claim_type", "A_CLAIM")
        res = c.get("result", "UNCERTAIN")
        if ctype == "A_CLAIM":
            if res == "GRANTED": scores["A_WIN"] += 2.0 * w; scores["PARTIAL_A_WIN"] += 1.0 * w
            elif res == "PARTIALLY_GRANTED": scores["PARTIAL_A_WIN"] += 2.0 * w
            elif res == "REJECTED": scores["B_WIN"] += 2.0 * w; scores["PARTIAL_B_WIN"] += 1.0 * w
        elif ctype in {"B_COUNTERCLAIM", "B_DEFENSE"}:
            if res == "GRANTED": scores["B_WIN"] += 2.0 * w; scores["PARTIAL_B_WIN"] += 1.0 * w
            elif res == "PARTIALLY_GRANTED": scores["PARTIAL_B_WIN"] += 1.5 * w
            elif res == "REJECTED": scores["A_WIN"] += 1.5 * w
    total = sum(scores.values())
    return {k: v / total for k, v in scores.items()}

def result_reliability(res: dict[str, Any], config: AuditorConfig) -> float:
    score = 0.6
    conf = res.get("confidence", "LOW")
    if conf == "HIGH": score += 0.2
    elif conf == "MEDIUM": score += 0.1
    claims = res.get("claims", [])
    if len(claims) >= config.min_valid_claims_for_full_weight:
        score += 0.1
    return min(1.0, score)

def consensus_probabilities(
    primary: dict[str, Any],
    auditor: dict[str, Any],
    config: AuditorConfig,
) -> dict[str, float]:
    p_dir = normalize_probabilities(primary.get("probabilities"), primary.get("prediction"))
    a_dir = normalize_probabilities(auditor.get("probabilities"), auditor.get("prediction"))
    p_claim = claim_table_probabilities(primary.get("claims", []))
    a_claim = claim_table_probabilities(auditor.get("claims", []))

    p_rel = result_reliability(primary, config)
    a_rel = result_reliability(auditor, config)

    w_p_dir = config.direct_primary_weight * p_rel
    w_p_claim = config.claim_primary_weight * p_rel
    w_a_dir = config.direct_auditor_weight * a_rel
    w_a_claim = config.claim_auditor_weight * a_rel

    total_w = w_p_dir + w_p_claim + w_a_dir + w_a_claim
    if total_w < 1e-6: total_w = 1.0

    combined = {}
    for lbl in LABELS:
        combined[lbl] = (
            w_p_dir * p_dir.get(lbl, 0.0) +
            w_p_claim * p_claim.get(lbl, 0.0) +
            w_a_dir * a_dir.get(lbl, 0.0) +
            w_a_claim * a_claim.get(lbl, 0.0)
        ) / total_w
    return combined

def run_auditor_pass(
    cases: Sequence[CaseBundle],
    runtime: TextAuditorRuntime,
    output_dir: Path,
    max_new_tokens: int = 950,
) -> dict[str, dict[str, Any]]:
    output_dir.mkdir(parents=True, exist_ok=True)
    results = {}
    for pos, case in enumerate(cases, start=1):
        path = output_dir / f"{case.case_id}.json"
        if path.exists():
            results[case.case_id] = read_json(path)
            continue

        user = f"""
ĐỊNH NGHĨA NHÃN:
{label_definition_text()}

CASE:
{case.packet_text}

Hãy phân rã claims rồi dự đoán.
{prediction_schema()}
"""
        print(f"[Auditor {pos}/{len(cases)}] Case: {case.case_id}")
        try:
            raw = runtime.generate_json(AUDITOR_SYSTEM, user, max_new_tokens)
            payload = parse_model_result(raw, case, runtime.model_id)
            payload["pass_name"] = "independent_auditor"
            print(f"   -> Predicted: {payload['prediction']} (confidence: {payload['confidence']}, claims: {len(payload['claims'])})")
        except Exception as exc:
            print(f"   ⚠️ [FALLBACK] Case {case.case_id} không parse được ({type(exc).__name__}: {exc}). Gán fallback.")
            traceback.print_exc()
            payload = {
                "case_id": case.case_id,
                "model_id": runtime.model_id,
                "pass_name": "independent_auditor",
                "prediction": "PARTIAL_A_WIN",
                "probabilities": {"A_WIN": 0.2, "PARTIAL_A_WIN": 0.35, "PARTIAL_B_WIN": 0.25, "B_WIN": 0.2},
                "claims": [],
                "confidence": "LOW",
                "decision_summary": "Fallback an toàn do lỗi parse",
                "error": f"{type(exc).__name__}: {exc}",
            }
        atomic_write_json(path, payload)
        results[case.case_id] = payload
    return results


Writing /content/alqac_auditor_standalone.py


In [ ]:
# =============================================================================
# 2. CẤU HÌNH ĐƯỜNG DẪN DỮ LIỆU VÀ NẠP SẴN SUBMISSION CỦA PRIMARY
# =============================================================================
from pathlib import Path
import sys, os, json
sys.path.insert(0, '/content')

POSSIBLE_DATA_DIRS = [
    Path('/content/data'),
    Path('/content/data/public_test'),
    Path('./ideal_retrieval_conditions/data'),
    Path('./data'),
]

DATA_DIR = None
for p in POSSIBLE_DATA_DIRS:
    if (p / 'corpus_law_pub.json').exists() or (p / 'public_test' / 'corpus_law_pub.json').exists():
        DATA_DIR = p if (p / 'corpus_law_pub.json').exists() else (p / 'public_test')
        break

if DATA_DIR is None:
    raise FileNotFoundError('Không tìm thấy thư mục chứa dữ liệu test và corpus! Hãy upload folder data vào /content/data')

PRIVATE_TEST_PATH = DATA_DIR / 'ALQAC_public_test.json'
LAW_CORPUS_PATH = DATA_DIR / 'corpus_law_pub.json'
CASE_RETRIEVAL_DIR = DATA_DIR / 'case_retrieval_outputs'
PRIVATE_CASE_AND_LAW_PATH = DATA_DIR / 'public_case_and_law_evidence.json'
GROUND_TRUTH_PATH = DATA_DIR / 'ground_truth.json' if (DATA_DIR / 'ground_truth.json').exists() else (DATA_DIR.parent / 'ground_truth.json')

WORK_DIR = Path('/content/submissions')
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Tìm và nạp file submission và predictions của Primary đã có từ trước (folder trace hoặc /content)
POSSIBLE_PRIMARY_FILES = [
    Path('/content/submissions/primary_predictions.json'),
    Path('/content/primary_predictions.json'),
    Path('./trace/primary_predictions.json'),
    Path('/content/data/primary_predictions.json'),
]

POSSIBLE_PRIMARY_SUB = [
    Path('/content/submissions/submission_primary.json'),
    Path('/content/submission_primary.json'),
    Path('./trace/submission_primary.json'),
]

PRIMARY_PRED_PATH = next((p for p in POSSIBLE_PRIMARY_FILES if p.exists()), None)
PRIMARY_SUB_PATH = next((p for p in POSSIBLE_PRIMARY_SUB if p.exists()), None)

print('--- ĐƯỜNG DẪN DỮ LIỆU ---')
print('Data dir:', DATA_DIR)
print('Test path:', PRIVATE_TEST_PATH)
print('Law corpus:', LAW_CORPUS_PATH)
print('Primary predictions:', PRIMARY_PRED_PATH)
print('Primary submission:', PRIMARY_SUB_PATH)
if PRIMARY_PRED_PATH is None or PRIMARY_SUB_PATH is None:
    print('⚠️ CHÚ Ý: Hãy đảm bảo bạn đã tải lên submission_primary.json và primary_predictions.json vào /content/submissions/')


--- ĐƯỜNG DẪN DỮ LIỆU ---
Data dir: /content/data/public_test
Test path: /content/data/public_test/ALQAC_public_test.json
Law corpus: /content/data/public_test/corpus_law_pub.json
Primary predictions: /content/submissions/primary_predictions.json
Primary submission: /content/submissions/submission_primary.json


In [ ]:
# =============================================================================
# 3. CẤU HÌNH AUDITOR MODEL
# =============================================================================
from getpass import getpass
from alqac_auditor_standalone import AuditorConfig

HF_TOKEN = os.getenv('HF_TOKEN', '').strip()
if not HF_TOKEN:
    print('Nếu muốn dùng CohereLabs/aya-expanse-8b, hãy nhập token Hugging Face (hoặc Enter để fallback sang Qwen2.5 / Qwen3):')
    HF_TOKEN = getpass('HF Token: ').strip()

CONFIG = AuditorConfig(
    # Model chính cho Auditor (Aya Expanse 8B)
    auditor_model_id='CohereLabs/aya-expanse-8b',
    # Model dự phòng miễn phí, không bị gated terms nếu không có HF token:
    auditor_fallback_model_id='Qwen/Qwen2.5-7B-Instruct',

    auditor_quant_mode='bf16',
    allow_auditor_fallback=True,
    hf_token=HF_TOKEN or None,

    deterministic_generation=True,
    auditor_max_new_tokens=2500,
    expected_private_cases=50,
)

print('Cấu hình Auditor:')
for k, v in CONFIG.__dict__.items():
    if k != 'hf_token':
        print(f'  {k}: {v}')


Nếu muốn dùng CohereLabs/aya-expanse-8b, hãy nhập token Hugging Face (hoặc Enter để fallback sang Qwen2.5 / Qwen3):
HF Token: ··········
Cấu hình Auditor:
  auditor_model_id: CohereLabs/aya-expanse-8b
  auditor_fallback_model_id: Qwen/Qwen2.5-7B-Instruct
  auditor_quant_mode: bf16
  allow_auditor_fallback: True
  deterministic_generation: True
  auditor_max_new_tokens: 2500
  max_chunks: 24
  max_words_per_chunk: 140
  max_total_evidence_words: 1900
  max_law_articles: 16
  max_words_per_law: 150
  max_total_law_words: 900
  max_hint_words: 230
  max_outcome_clues: 10
  direct_primary_weight: 0.32
  claim_primary_weight: 0.22
  direct_auditor_weight: 0.27
  claim_auditor_weight: 0.19
  min_valid_claims_for_full_weight: 2
  evidence_citation_bonus: 0.08
  law_citation_bonus: 0.05
  disagreement_claim_rule_bonus: 0.08
  expected_private_cases: 50


In [ ]:
# =============================================================================
# 4. TIẾN HÀNH CHẠY AUDITOR PREDICTION (ĐỘC LẬP)
# =============================================================================
import time
from collections import Counter
from alqac_auditor_standalone import (
    read_json,
    atomic_write_json,
    load_article_corpus,
    load_case_retrieval,
    load_private_case_and_law,
    build_case_bundles,
    TextAuditorRuntime,
    run_auditor_pass,
)

print('=== Đang nạp corpus và dựng Case Bundles ===')
test_cases = read_json(PRIVATE_TEST_PATH)
article_corpus = load_article_corpus(LAW_CORPUS_PATH)
retrieval_data = load_case_retrieval(CASE_RETRIEVAL_DIR)
case_law_data = load_private_case_and_law(PRIVATE_CASE_AND_LAW_PATH)

bundles = build_case_bundles(
    test_cases,
    retrieval_data,
    case_law_data,
    article_corpus,
    CONFIG,
)
print(f'Đã tạo {len(bundles)} case bundles.')

# Khởi tạo và load Auditor Runtime
auditor_runtime = TextAuditorRuntime(CONFIG)
auditor_runtime.load()

# Chạy Auditor Pass
auditor_preds_dir = WORK_DIR / 'auditor_predictions_cache'
print(f'\n=== Bắt đầu chạy Auditor Pass ({len(bundles)} cases) ===')
t0 = time.time()
auditor_results = run_auditor_pass(bundles, auditor_runtime, auditor_preds_dir, CONFIG.auditor_max_new_tokens)
t_elapsed = time.time() - t0
print(f'\nHoàn thành Auditor Pass trong {t_elapsed:.1f}s.')

# Giải phóng VRAM Auditor ngay lập tức
auditor_runtime.unload()
print('Đã giải phóng VRAM của Auditor.')

# Lưu file tổng hợp auditor_predictions.json
auditor_predictions_path = WORK_DIR / 'auditor_predictions.json'
atomic_write_json(auditor_predictions_path, auditor_results)
print('Đã lưu auditor_predictions:', auditor_predictions_path)

# In phân phối nhãn của Auditor vừa chạy
auditor_label_counts = Counter(v['prediction'] for v in auditor_results.values())
print('\nPhân phối nhãn thực tế của Auditor:')
for lbl, cnt in auditor_label_counts.items():
    print(f'  {lbl}: {cnt} ({cnt/len(auditor_results)*100:.1f}%)')


=== Đang nạp corpus và dựng Case Bundles ===
Đã tạo 50 case bundles.
Đang tải tokenizer/processor cho CohereLabs/aya-expanse-8b...
AutoTokenizer warning: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/CohereLabs/aya-expanse-8b.
401 Client Error. (Request ID: Root=1-6accb806-34d6346a0a51f3f43d28d805;25838918-c649-4fca-b9df-d2aae2ec5932)

Cannot access gated repo for url https://huggingface.co/CohereLabs/aya-expanse-8b/resolve/5062468bf9bc0c6035fd64e06274333ec127d980/config.json.
Access to model CohereLabs/aya-expanse-8b is restricted. You must have access to it and be authenticated to access it. Please log in., thử AutoProcessor...
Không load được primary auditor (CohereLabs/aya-expanse-8b): OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/CohereLabs/aya-expanse-8b.
401 Client Error. (Request ID: Root=1-6accb807-297c09b22507e6c327df0e63;612640ae-0751-4d2f-b71b-38bbe0cc0322)

Cannot a

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Auditor loaded successfully: Qwen/Qwen2.5-7B-Instruct (fallback=True, device=cuda:0)

=== Bắt đầu chạy Auditor Pass (50 cases) ===
[Auditor 1/50] Case: case_4101
   -> Predicted: A_WIN (confidence: HIGH, claims: 1)
[Auditor 2/50] Case: case_4337
   -> Predicted: PARTIAL_A_WIN (confidence: MEDIUM, claims: 1)
[Auditor 3/50] Case: case_4588
   -> Predicted: PARTIAL_A_WIN (confidence: MEDIUM, claims: 1)
[Auditor 4/50] Case: case_4616
   -> Predicted: B_WIN (confidence: HIGH, claims: 3)
[Auditor 5/50] Case: case_6616
   -> Predicted: PARTIAL_A_WIN (confidence: MEDIUM, claims: 1)
[Auditor 6/50] Case: case_5226
   -> Predicted: B_WIN (confidence: MEDIUM, claims: 1)
[Auditor 7/50] Case: case_6284
   -> Predicted: PARTIAL_A_WIN (confidence: MEDIUM, claims: 1)
[Auditor 8/50] Case: case_3995
   -> Predicted: PARTIAL_A_WIN (confidence: MEDIUM, claims: 1)
[Auditor 9/50] Case: case_6384
   -> Predicted: PARTIAL_A_WIN (confidence: MEDIUM, claims: 2)
[Auditor 10/50] Case: case_8219
   -> Predicted: PA

In [ ]:
# =============================================================================
# 5. TẠO SUBMISSION CHO AUDITOR & ENSEMBLE CONSENSUS VỚI PRIMARY
# =============================================================================
import json
from collections import Counter
from alqac_auditor_standalone import consensus_probabilities, top_label

primary_results = read_json(PRIMARY_PRED_PATH)
primary_sub = read_json(PRIMARY_SUB_PATH)
primary_sub_map = {row['case_id']: row for row in primary_sub}

# 1. Xuất file submission_auditor.json (độc lập)
auditor_rows = []
for tc in test_cases:
    cid = str(tc['case_id']).strip()
    p_sub = primary_sub_map.get(cid, {})
    aud_pred = auditor_results.get(cid, {}).get('prediction', 'PARTIAL_A_WIN')
    auditor_rows.append({
        'case_id': cid,
        'prediction': aud_pred,
        'case_evidence': p_sub.get('case_evidence', []),
        'law_evidence': p_sub.get('law_evidence', []),
    })

auditor_submission_path = WORK_DIR / 'submission_auditor.json'
atomic_write_json(auditor_submission_path, auditor_rows)
print('Đã lưu:', auditor_submission_path)

# 2. Xuất file ensemble submission (Consensus giữa Primary và Auditor)
ensemble_rows = []
ensemble_diagnostics = []

for tc in test_cases:
    cid = str(tc['case_id']).strip()
    p_res = primary_results.get(cid, {})
    a_res = auditor_results.get(cid, {})
    p_sub = primary_sub_map.get(cid, {})

    probs = consensus_probabilities(p_res, a_res, CONFIG)
    ens_pred = top_label(probs)

    ensemble_rows.append({
        'case_id': cid,
        'prediction': ens_pred,
        'case_evidence': p_sub.get('case_evidence', []),
        'law_evidence': p_sub.get('law_evidence', []),
    })
    ensemble_diagnostics.append({
        'case_id': cid,
        'primary_prediction': p_res.get('prediction'),
        'auditor_prediction': a_res.get('prediction'),
        'ensemble_prediction': ens_pred,
        'probabilities': probs,
    })

ensemble_submission_path = WORK_DIR / 'submission_ensemble_primary_auditor.json'
atomic_write_json(ensemble_submission_path, ensemble_rows)
print('Đã lưu:', ensemble_submission_path)

primary_counts = Counter(row['prediction'] for row in primary_sub)
auditor_counts = Counter(row['prediction'] for row in auditor_rows)
ensemble_counts = Counter(row['prediction'] for row in ensemble_rows)

print('\n' + '='*65)
print('SO SÁNH PHÂN PHỐI NHÃN (50 CASES)')
print('='*65)
for lbl in ['A_WIN', 'PARTIAL_A_WIN', 'PARTIAL_B_WIN', 'B_WIN']:
    print(f'{lbl:<15}: Primary={primary_counts.get(lbl, 0):2d} | Auditor={auditor_counts.get(lbl, 0):2d} | Ensemble={ensemble_counts.get(lbl, 0):2d}')
print('='*65)


Đã lưu: /content/submissions/submission_auditor.json
Đã lưu: /content/submissions/submission_ensemble_primary_auditor.json

SO SÁNH PHÂN PHỐI NHÃN (50 CASES)
A_WIN          : Primary= 6 | Auditor=11 | Ensemble= 5
PARTIAL_A_WIN  : Primary=37 | Auditor=36 | Ensemble=38
PARTIAL_B_WIN  : Primary= 2 | Auditor= 1 | Ensemble= 1
B_WIN          : Primary= 5 | Auditor= 2 | Ensemble= 6


In [ ]:
# =============================================================================
# 6. ĐÁNH GIÁ ACCURACY TRÊN TẬP PUBLIC TEST (GROUND TRUTH)
# =============================================================================
import pandas as pd

if GROUND_TRUTH_PATH and GROUND_TRUTH_PATH.exists():
    gt_data = read_json(GROUND_TRUTH_PATH)
    gt_map = {item['case_id']: item['verdict_label'] for item in gt_data}

    p_map = {r['case_id']: r['prediction'] for r in primary_sub}
    a_map = {r['case_id']: r['prediction'] for r in auditor_rows}
    e_map = {r['case_id']: r['prediction'] for r in ensemble_rows}

    total = len(gt_data)
    correct_p = sum(1 for cid, gt in gt_map.items() if p_map.get(cid) == gt)
    correct_a = sum(1 for cid, gt in gt_map.items() if a_map.get(cid) == gt)
    correct_e = sum(1 for cid, gt in gt_map.items() if e_map.get(cid) == gt)

    print('\n' + '='*65)
    print(f'KẾT QUẢ ĐÁNH GIÁ ACCURACY TRÊN {total} PUBLIC CASES')
    print('='*65)
    print(f'1. Primary (Qwen3.5-9B)               : {correct_p}/{total} = {correct_p/total*100:.2f}%')
    print(f'2. Auditor ({CONFIG.auditor_model_id if not auditor_runtime.using_fallback else CONFIG.auditor_fallback_model_id}) : {correct_a}/{total} = {correct_a/total*100:.2f}%')
    print(f'3. Ensemble Consensus (Primary+Auditor): {correct_e}/{total} = {correct_e/total*100:.2f}%')
    print('='*65)

    eval_rows = []
    for cid, gt in gt_map.items():
        pred_p = p_map.get(cid, '')
        pred_a = a_map.get(cid, '')
        pred_e = e_map.get(cid, '')
        eval_rows.append({
            'case_id': cid,
            'ground_truth': gt,
            'primary': pred_p,
            'primary_ok': '✓' if pred_p == gt else '✗',
            'auditor': pred_a,
            'auditor_ok': '✓' if pred_a == gt else '✗',
            'ensemble': pred_e,
            'ensemble_ok': '✓' if pred_e == gt else '✗',
        })
    df_eval = pd.DataFrame(eval_rows)
    display(df_eval.head(20))
else:
    print('Không có file ground_truth.json (private test). Bỏ qua bước tính accuracy.')



KẾT QUẢ ĐÁNH GIÁ ACCURACY TRÊN 50 PUBLIC CASES
1. Primary (Qwen3.5-9B)               : 20/50 = 40.00%
2. Auditor (Qwen/Qwen2.5-7B-Instruct) : 13/50 = 26.00%
3. Ensemble Consensus (Primary+Auditor): 20/50 = 40.00%


,case_id,ground_truth,primary,primary_ok,auditor,auditor_ok,ensemble,ensemble_ok
0,case_4101,PARTIAL_A_WIN,PARTIAL_A_WIN,✓,A_WIN,✗,PARTIAL_A_WIN,✓
1,case_4337,A_WIN,A_WIN,✓,PARTIAL_A_WIN,✗,A_WIN,✓
2,case_4588,A_WIN,PARTIAL_A_WIN,✗,PARTIAL_A_WIN,✗,PARTIAL_A_WIN,✗
3,case_4616,A_WIN,PARTIAL_A_WIN,✗,B_WIN,✗,B_WIN,✗
4,case_6616,A_WIN,PARTIAL_A_WIN,✗,PARTIAL_A_WIN,✗,PARTIAL_A_WIN,✗
5,case_5226,A_WIN,PARTIAL_A_WIN,✗,B_WIN,✗,PARTIAL_A_WIN,✗
6,case_6284,A_WIN,PARTIAL_A_WIN,✗,PARTIAL_A_WIN,✗,PARTIAL_A_WIN,✗
7,case_3995,A_WIN,PARTIAL_A_WIN,✗,PARTIAL_A_WIN,✗,PARTIAL_A_WIN,✗
8,case_6384,A_WIN,PARTIAL_A_WIN,✗,PARTIAL_A_WIN,✗,PARTIAL_A_WIN,✗
9,case_8219,A_WIN,A_WIN,✓,PARTIAL_A_WIN,✗,PARTIAL_A_WIN,✗


In [ ]:
# =============================================================================
# 7. ĐÓNG GÓI VÀ DOWNLOAD CÁC FILE SUBMISSION
# =============================================================================
import zipfile

zip_path = Path('/content/submissions_auditor_ensemble.zip')
files_to_zip = [
    auditor_submission_path,
    ensemble_submission_path,
    auditor_predictions_path,
]
if PRIMARY_SUB_PATH and PRIMARY_SUB_PATH.exists():
    files_to_zip.append(PRIMARY_SUB_PATH)
if PRIMARY_PRED_PATH and PRIMARY_PRED_PATH.exists():
    files_to_zip.append(PRIMARY_PRED_PATH)

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for f in files_to_zip:
        if f.exists():
            zipf.write(f, arcname=f.name)

print(f'✅ Đã nén các file submission vào: {zip_path} ({zip_path.stat().st_size / 1024:.1f} KB)')
print('Các file có trong zip:')
for f in files_to_zip:
    if f.exists():
        print(f'  - {f.name}')

try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    pass


✅ Đã nén các file submission vào: /content/submissions_auditor_ensemble.zip (114.5 KB)
Các file có trong zip:
  - submission_auditor.json
  - submission_ensemble_primary_auditor.json
  - auditor_predictions.json
  - submission_primary.json
  - primary_predictions.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>